# Industry Project: From Messy Notebook Code to Production Python (Production-Style)

> **Who is this notebook for?** Someone who can write Python in a notebook, but has never turned that code into something a team can run, test and trust.
> You do **not** need to know packaging, testing or CLIs already. Every step is explained in plain English, and almost every line of code has a comment.

---

### How this notebook is different from `02_industrial.ipynb`

| `02_industrial.ipynb` | This notebook |
|---|---|
| Shows **each pattern on its own** (logging, pytest, dataclasses, pyproject...) with small snippets | Takes **one real messy script** and refactors it end-to-end, step by step |
| Tests are written as functions and called by hand; `pyproject.toml` is printed as a string | Real files are written to disk (`src/` layout), the **CLI is executed** with `subprocess`, **pytest really runs**, a wheel is really built |
| "Never use print in production" | Shows *where* print is still right (program output on stdout) and where logging is (diagnostics on stderr) |
| Config from env and YAML shown separately | One `TrainConfig` dataclass with **defaults < YAML/JSON file < environment variables**, validated, with clear errors |
| - | Custom exceptions mapped to **exit codes**, so schedulers (cron, Airflow) know what failed |
| - | A **code-review checklist** and a tiny automated reviewer that compares "before" and "after" |

---

### The business problem

A data scientist at a subscription company wrote `train_model.py` by exporting a Jupyter notebook. It trains a customer-churn model. Every Monday a colleague runs it by hand. Problems the team keeps hitting:

- it only works on the author's laptop (hard-coded file names, must be run from the right folder);
- when the data export changed a column name, the script **still printed "saved"** and shipped a broken model for three weeks;
- nobody dares to change it, because there are **no tests**;
- the scheduler (cron/Airflow) cannot tell success from failure: the script **always exits with code 0**.

**Your job:** turn it into a small, maintainable Python package with a command-line interface, configuration, logging, error handling and tests, *without changing what the model does*. Success = the same model quality, but the job fails loudly on bad input, can be configured without editing code, and every change is checked by tests in seconds.

---

### The refactoring workflow we will follow

```
 1. Run the messy script (it works... until it doesn't)  ->  2. List the code smells
            |
 3. Plan the src/ layout  ->  4. exceptions  ->  5. config  ->  6. data + features  ->  7. model
            |
 8. CLI (argparse) + logging + exit codes  ->  9. pyproject.toml  ->  10. run the CLI for real
            |
 11. pytest  ->  12. build a wheel  ->  13. automated before/after review  ->  14. code-review checklist
```

### Table of Contents
1. Setup: a temporary workspace and two helpers
2. The data: a weekly CRM export (simulated)
3. The "before" script, and how it fails silently
4. Code smells: what is wrong and why it matters
5. Target layout: the `src/` project structure
6. Custom exceptions
7. Configuration: dataclass + YAML/JSON + environment variables
8. Data loading, validation, cleaning and features as small typed functions
9. The model module: train, evaluate, save, load
10. The command-line interface: argparse, logging, exit codes
11. Packaging metadata: `pyproject.toml`
12. Running the CLI like a scheduler would
13. Unit tests with pytest (really executed)
14. Building a wheel (proof that the package is installable)
15. Automated before/after review
16. Code-review checklist
17. Summary, exercises and resources

---
## 1. Setup: a temporary workspace and two helpers

We will create a **real project on disk**, but in a temporary folder, so this notebook never writes into the course repository. In real life this folder would be your git repository.

Two helpers:
- `write_file(path, text)`: writes a file into the workspace. It does what Jupyter's `%%writefile` magic does, but with `pathlib`, so it works anywhere (also in scripts).
- `run(command)`: runs a command in a **separate process**, exactly like you would in a terminal, and shows its exit code, its output (`stdout`) and its log messages (`stderr`).

In [1]:
import ast                              # parse Python source code (for the automated review in Section 15)
import importlib.util                   # check whether a package (pytest) is installed
import json                             # read the metrics file our CLI writes
import logging                          # status messages for this notebook
import os                               # environment variables for the subprocesses
import shutil                           # copy files
import subprocess                       # run the CLI / pytest as separate processes, like a terminal
import sys                              # sys.executable = the Python of this notebook's environment
import tempfile                         # temporary workspace (never write into the repo)
import tomllib                          # read TOML files (standard library since Python 3.11)
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
log = logging.getLogger("refactor")

WORKDIR = Path(tempfile.mkdtemp(prefix="churnkit_project_"))   # real life: your git repository
PYTHON = sys.executable                                          # same interpreter/venv as this notebook


def write_file(relative_path: str, content: str) -> Path:
    """Create/overwrite a file inside the workspace (a pathlib version of %%writefile)."""
    path = WORKDIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)               # create folders as needed
    path.write_text(content.lstrip("\n"), encoding="utf-8")
    log.info("wrote %-32s (%d lines)", relative_path, len(content.strip().splitlines()))
    return path


def run(args: list[str], cwd: Path = WORKDIR, extra_env: dict | None = None, quiet: bool = False):
    """Run a command in a fresh process and print exit code, stdout and stderr (paths shortened)."""
    env = {k: v for k, v in os.environ.items() if not k.startswith("CHURNKIT_")}   # clean slate
    env.update({"PYTHONPATH": str(WORKDIR / "src"),              # real life: `pip install -e .` instead
                "OMP_NUM_THREADS": "1", **(extra_env or {})})
    result = subprocess.run(args, cwd=cwd, env=env, capture_output=True, text=True, timeout=900)
    shorten = lambda text: text.replace(str(WORKDIR), "<workdir>").replace(PYTHON, "python")
    shown = " ".join([f"{k}={v}" for k, v in (extra_env or {}).items()] + [shorten(a) for a in args])
    print(f"$ {shown}\n-> exit code {result.returncode}")
    if not quiet:
        if result.stdout.strip():
            print("[stdout]\n" + shorten(result.stdout.rstrip()))
        if result.stderr.strip():
            print("[stderr]\n" + shorten(result.stderr.rstrip()))
    return result


print("Workspace:", WORKDIR)
print("Python   :", PYTHON)

Workspace: /tmp/churnkit_project_ubybwvmw
Python   : /path/to/.venv/bin/python


---
## 2. The data: a weekly CRM export (simulated)

The script expects a CSV export of customers. We simulate three files:
- `customers.csv`: this week's training data, with the usual mess (duplicate rows, inconsistent spelling like `" PRO "`, missing and negative fees);
- `customers_renamed.csv`: the same data after the CRM team **renamed a column** (`tenure_months` -> `tenure_mo`), the change that broke production;
- `new_customers.csv`: customers to score (no `churned` column yet).

In [2]:
def make_customers(n: int = 1500, seed: int = 7, with_target: bool = True) -> pd.DataFrame:
    """Simulated CRM export with realistic mess."""
    rng = np.random.default_rng(seed)
    plan = rng.choice(["basic", "pro", "enterprise"], n, p=[0.5, 0.35, 0.15])
    tenure = rng.integers(1, 60, n)
    users = rng.integers(1, 20, n)
    fee = (np.select([plan == "basic", plan == "pro"], [20, 60], 150) + rng.normal(0, 8, n)).round(2)
    calls = rng.poisson(1.5, n)
    logit = -0.6 + 0.45 * calls - 0.05 * tenure + 0.6 * (plan == "basic") - 0.02 * users
    df = pd.DataFrame({"customer_id": [f"C{seed}{i:05d}" for i in range(n)], "plan": plan,
                       "monthly_fee": fee, "tenure_months": tenure, "users": users, "support_calls": calls})
    if with_target:
        df["churned"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    # --- the usual mess ---
    idx = rng.choice(n, 120, replace=False)
    df.loc[idx, "plan"] = " " + df.loc[idx, "plan"].str.upper() + " "          # " PRO " instead of "pro"
    df.loc[rng.choice(n, 40, replace=False), "monthly_fee"] = np.nan            # missing values
    df.loc[rng.choice(n, 8, replace=False), "monthly_fee"] = -1.0               # billing bug
    return pd.concat([df, df.sample(25, random_state=seed)], ignore_index=True)  # duplicated rows


data_dir = WORKDIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)
customers = make_customers()
customers.to_csv(data_dir / "customers.csv", index=False)
customers.rename(columns={"tenure_months": "tenure_mo"}).to_csv(data_dir / "customers_renamed.csv", index=False)
make_customers(n=300, seed=11, with_target=False).to_csv(data_dir / "new_customers.csv", index=False)

print(f"customers.csv: {len(customers):,} rows | churn rate {customers['churned'].mean():.1%}")
customers.head()

customers.csv: 1,525 rows | churn rate 25.7%


,customer_id,plan,monthly_fee,tenure_months,users,support_calls,churned
0,C700000,pro,53.31,32,9,3,0
1,C700001,enterprise,144.50,39,6,2,0
2,C700002,pro,70.94,19,7,0,0
3,C700003,basic,22.46,43,18,2,0
4,C700004,basic,18.15,51,5,2,0


---
## 3. The "before" script, and how it fails silently

This is a typical notebook export. Read it once; it *looks* fine and it *does* work on the happy path.

In [3]:
# The original script, exported from 'Untitled3.ipynb' (do NOT copy this style)
_ = write_file('legacy/train_model.py', r'''
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pickle

df = pd.read_csv("customers.csv")
print(df.shape)
df = df.drop_duplicates()
df['plan'] = df['plan'].str.lower().str.strip()
df['monthly_fee'] = df['monthly_fee'].fillna(df['monthly_fee'].mean())
df = df[df['monthly_fee'] > 0]
try:
    df['tenure_years'] = df['tenure_months'] / 12
except:
    df['tenure_years'] = 0
df['fee_per_user'] = df['monthly_fee'] / df['users']
X = pd.get_dummies(df[['plan', 'monthly_fee', 'tenure_years', 'fee_per_user', 'support_calls']])
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
m = LogisticRegression(C=0.5, max_iter=500)
m.fit(X_train, y_train)
acc = (m.predict(X_test) == y_test).mean()
print("acc", acc)
if acc > 0.5:
    pickle.dump(m, open("model.pkl", "wb"))
    print("saved")
''')

2026-10-21 17:43:15,671 | INFO | wrote legacy/train_model.py            (27 lines)


In [4]:
# Happy path: run it from the data folder (it only works from there, because of the relative path)
before_ok = run([PYTHON, str(WORKDIR / "legacy/train_model.py")], cwd=data_dir)

$ python <workdir>/legacy/train_model.py
-> exit code 0
[stdout]
(1525, 7)
acc 0.7248322147651006
saved


In [5]:
# Now the CRM team renames a column. We simulate next week's export in a fresh folder.
next_week = WORKDIR / "next_week"
next_week.mkdir(exist_ok=True)
shutil.copy(data_dir / "customers_renamed.csv", next_week / "customers.csv")
before_bad = run([PYTHON, str(WORKDIR / "legacy/train_model.py")], cwd=next_week)

$ python <workdir>/legacy/train_model.py
-> exit code 0
[stdout]
(1525, 7)
acc 0.6946308724832215
saved


**This is the worst kind of bug: a silent failure.** The column `tenure_months` no longer exists, the bare `except:` swallowed the `KeyError`, set `tenure_years = 0` for everyone, and the script still printed `saved` with **exit code 0**. The scheduler is happy and the model quietly lost an important feature. Accuracy dropped by about 3 points (0.72 to 0.69), the kind of change nobody notices in a log line. Worse: **both** numbers are below the ~0.74 you would get by always predicting "no churn" (about 26% of customers churn), yet the script's `acc > 0.5` "quality gate" happily passed.

---
## 4. Code smells: what is wrong and why it matters

| # | Smell in the script | Why it hurts in production | Fix in the refactor |
|---|---|---|---|
| 1 | Everything runs at import time, no functions | Cannot test or reuse a single step | Small, typed functions in modules |
| 2 | Hard-coded `"customers.csv"`, `"model.pkl"`, `C=0.5`, `0.2` | Must edit code to change settings; works only from one folder | `TrainConfig` dataclass + YAML/JSON file + env vars |
| 3 | `print()` for status | No timestamps/levels; mixes diagnostics with results | `logging` (stderr) + one machine-readable result line (stdout) |
| 4 | Bare `except:` that hides errors | Silent failures (we just saw one) | Validate inputs and raise **specific custom exceptions** |
| 5 | Always exits with code 0 | Schedulers cannot detect failures | `main()` returns **exit codes** per error type |
| 6 | Mean imputation on all data before the split; `get_dummies` | Leakage; train/serve column mismatch on new data | sklearn `Pipeline` with imputer + `OneHotEncoder(handle_unknown="ignore")` |
| 7 | `pickle` of the bare model, no metadata | Nobody knows which data/config/version produced it | `joblib` + `metrics.json` with config and versions |
| 8 | Quality gate on accuracy > 0.5 | Passes even for a useless model on imbalanced data | Configurable ROC-AUC gate that raises `ModelQualityError` |
| 9 | No tests | Every change is scary | `pytest` suite: unit + end-to-end tests |

**Golden rule of refactoring:** change the *structure*, not the *behaviour*, in small steps, and keep the result runnable after every step. Big-bang rewrites are how teams lose months.

---
## 5. Target layout: the `src/` project structure

```
churnkit/                      <- project root (your git repository)
├── pyproject.toml             <- name, version, dependencies, CLI entry point, tool settings
├── configs/
│   └── train.yaml             <- settings that change without code changes
├── src/
│   └── churnkit/              <- the importable package
│       ├── __init__.py        <- version
│       ├── __main__.py        <- enables `python -m churnkit`
│       ├── exceptions.py      <- our error types
│       ├── config.py          <- TrainConfig + load_config()
│       ├── data.py            <- load / validate / clean
│       ├── features.py        <- feature engineering
│       ├── model.py           <- train / evaluate / save / load / predict
│       └── cli.py             <- argparse, logging setup, exit codes
└── tests/
    ├── conftest.py            <- shared test fixtures
    ├── test_config.py
    ├── test_data.py
    └── test_cli.py
```

**Why `src/`?** With the package inside `src/`, Python cannot accidentally import your local folder instead of the installed package, so tests check what users will actually install. The Python Packaging User Guide recommends it.

**Dependency direction:** `cli` -> `model` -> `features` / `data` -> `config` / `exceptions`. Lower layers never import `cli`. This keeps every module testable on its own.

---
## 6. Custom exceptions

A **custom exception hierarchy** lets callers react to *kinds* of failure: a config typo is fixed differently from a broken data export. All our errors inherit from one base class, so a caller can also catch "anything from churnkit" in one line.

In [6]:
# Package marker + single source of truth for the version
_ = write_file('src/churnkit/__init__.py', r'''
"""churnkit: train and run the customer-churn model."""

__version__ = "0.1.0"
''')

2026-10-21 17:43:25,531 | INFO | wrote src/churnkit/__init__.py         (3 lines)


In [7]:
# One base class, one subclass per kind of failure
_ = write_file('src/churnkit/exceptions.py', r'''
"""Error types raised by churnkit. The CLI maps each one to an exit code."""


class ChurnKitError(Exception):
    """Base class for every error raised on purpose by churnkit."""


class ConfigError(ChurnKitError):
    """The configuration is missing, malformed or has invalid values."""


class DataValidationError(ChurnKitError):
    """The input data does not respect the expected contract."""


class ArtifactNotFoundError(ChurnKitError):
    """A model or metadata file that should exist is missing."""


class ModelQualityError(ChurnKitError):
    """The trained model is not good enough to be saved."""
''')

2026-10-21 17:43:25,556 | INFO | wrote src/churnkit/exceptions.py       (21 lines)


---
## 7. Configuration: dataclass + YAML/JSON + environment variables

All settings live in one **frozen dataclass**. Values can come from three places, each overriding the previous one (the same precedence most tools use, see the "Twelve-Factor App" principle *store config in the environment*):

```
defaults in the dataclass   <   config file (YAML or JSON)   <   environment variables CHURNKIT_*
```

- The file is for settings you **review in git** (e.g. regularisation strength).
- Environment variables are for settings that **differ per machine or run** (paths, a one-off experiment), and for anything secret (never put secrets in git).
- `__post_init__` validates values, so a typo like `test_size: 2` fails **immediately with a clear message**, not 20 minutes into a run.

In [8]:
# Typed configuration with validation and three sources
_ = write_file('src/churnkit/config.py', r'''
"""Configuration: defaults < YAML/JSON file < CHURNKIT_* environment variables."""

import json
import os
from dataclasses import dataclass, fields
from pathlib import Path
from typing import Any, Mapping

import yaml

from churnkit.exceptions import ConfigError

ENV_PREFIX = "CHURNKIT_"


@dataclass(frozen=True)
class TrainConfig:
    """All settings of a training run. frozen=True: nobody can change them by accident mid-run."""

    data_path: Path = Path("data/customers.csv")
    output_dir: Path = Path("artifacts")
    test_size: float = 0.2
    random_state: int = 42
    C: float = 1.0
    max_iter: int = 1000
    min_auc: float = 0.65

    def __post_init__(self) -> None:
        if not 0.0 < self.test_size < 1.0:
            raise ConfigError(f"test_size must be between 0 and 1, got {self.test_size}")
        if self.C <= 0:
            raise ConfigError(f"C must be positive, got {self.C}")
        if self.max_iter < 1:
            raise ConfigError(f"max_iter must be >= 1, got {self.max_iter}")
        if not 0.5 <= self.min_auc <= 1.0:
            raise ConfigError(f"min_auc must be between 0.5 and 1.0, got {self.min_auc}")


def _read_config_file(path: Path) -> dict[str, Any]:
    """Read a flat YAML or JSON mapping."""
    if not path.exists():
        raise ConfigError(f"Config file not found: {path}")
    text = path.read_text(encoding="utf-8")
    try:
        if path.suffix in {".yaml", ".yml"}:
            data = yaml.safe_load(text) or {}
        elif path.suffix == ".json":
            data = json.loads(text)
        else:
            raise ConfigError(f"Unsupported config format '{path.suffix}' (use .yaml, .yml or .json)")
    except (yaml.YAMLError, json.JSONDecodeError) as exc:
        raise ConfigError(f"Cannot parse {path}: {exc}") from exc
    if not isinstance(data, dict):
        raise ConfigError(f"{path} must contain a mapping of setting names to values")
    return data


def _convert(name: str, value: Any, target_type: type) -> Any:
    """Convert strings from env/YAML to the declared field type, with a clear error."""
    try:
        return target_type(value)
    except (TypeError, ValueError) as exc:
        raise ConfigError(f"Setting '{name}': cannot convert {value!r} to {target_type.__name__}") from exc


def load_config(path: Path | str | None = None, env: Mapping[str, str] | None = None) -> TrainConfig:
    """Build a TrainConfig from defaults, an optional file and environment variables."""
    env = os.environ if env is None else env
    field_types = {f.name: f.type for f in fields(TrainConfig)}
    values: dict[str, Any] = {}
    if path is not None:
        values.update(_read_config_file(Path(path)))
    unknown = sorted(set(values) - set(field_types))
    if unknown:
        raise ConfigError(f"Unknown setting(s) {unknown}; allowed: {sorted(field_types)}")
    for name in field_types:
        env_key = ENV_PREFIX + name.upper()
        if env_key in env:
            values[name] = env[env_key]
    converted = {name: _convert(name, value, field_types[name]) for name, value in values.items()}
    return TrainConfig(**converted)
''')

2026-10-21 17:43:25,585 | INFO | wrote src/churnkit/config.py           (81 lines)


In [9]:
# Try the config module directly from the notebook (a quick interactive check before writing tests)
sys.path.insert(0, str(WORKDIR / "src"))          # make `import churnkit` find our package
from churnkit.config import load_config           # noqa: E402  (import after sys.path change)
from churnkit.exceptions import ConfigError       # noqa: E402

yaml_path = write_file("configs/train.yaml", """
# Reviewed in git. Paths are relative to the project root.
data_path: data/customers.csv
output_dir: artifacts/churn/v1
C: 0.5
test_size: 0.2
""")

print("1. defaults only        :", load_config(env={}))
print("2. + YAML file          :", load_config(yaml_path, env={}))
print("3. + env var override   :", load_config(yaml_path, env={"CHURNKIT_C": "2.0"}).C, "(C from CHURNKIT_C)")
for bad_env in [{"CHURNKIT_TEST_SIZE": "1.5"}, {"CHURNKIT_MAX_ITER": "many"}]:
    try:
        load_config(yaml_path, env=bad_env)
    except ConfigError as err:
        print(f"4. rejected {bad_env}: {err}")

2026-10-21 17:43:25,674 | INFO | wrote configs/train.yaml               (5 lines)


1. defaults only        : TrainConfig(data_path=PosixPath('data/customers.csv'), output_dir=PosixPath('artifacts'), test_size=0.2, random_state=42, C=1.0, max_iter=1000, min_auc=0.65)
2. + YAML file          : TrainConfig(data_path=PosixPath('data/customers.csv'), output_dir=PosixPath('artifacts/churn/v1'), test_size=0.2, random_state=42, C=0.5, max_iter=1000, min_auc=0.65)
3. + env var override   : 2.0 (C from CHURNKIT_C)
4. rejected {'CHURNKIT_TEST_SIZE': '1.5'}: test_size must be between 0 and 1, got 1.5
4. rejected {'CHURNKIT_MAX_ITER': 'many'}: Setting 'max_iter': cannot convert 'many' to int


---
## 8. Data loading, validation, cleaning and features as small typed functions

Each function does **one thing**, has **type hints**, a **docstring**, and **never modifies its input** (it returns a new DataFrame). Small pure functions are easy to test: give an input, check the output.

The big change compared with the script: `validate_columns` **refuses** data that breaks the contract, instead of guessing. This is the fix for the silent failure in Section 3.

In [10]:
# Load -> validate -> clean. Each step is a separate, testable function
_ = write_file('src/churnkit/data.py', r'''
"""Loading, validating and cleaning the CRM customer export."""

import logging
from pathlib import Path

import numpy as np
import pandas as pd

from churnkit.exceptions import DataValidationError

logger = logging.getLogger(__name__)      # library code only creates a logger; the CLI configures it

TARGET = "churned"
REQUIRED_COLUMNS = ("customer_id", "plan", "monthly_fee", "tenure_months", "users", "support_calls", TARGET)
VALID_PLANS = frozenset({"basic", "pro", "enterprise"})


def load_customers(path: Path) -> pd.DataFrame:
    """Read the CSV export. Raises DataValidationError if the file is missing."""
    path = Path(path)
    if not path.exists():
        raise DataValidationError(f"Input file not found: {path}")
    df = pd.read_csv(path)
    logger.info("Loaded %d rows x %d columns from %s", len(df), df.shape[1], path.name)
    return df


def validate_columns(df: pd.DataFrame, require_target: bool = True) -> None:
    """Fail loudly if required columns are missing (the contract with the CRM team)."""
    required = [c for c in REQUIRED_COLUMNS if require_target or c != TARGET]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise DataValidationError(f"Missing required column(s) {missing}; got {sorted(df.columns)}")
    if require_target and not set(df[TARGET].dropna().unique()) <= {0, 1}:
        raise DataValidationError(f"Column '{TARGET}' must contain only 0/1")


def clean_customers(df: pd.DataFrame) -> pd.DataFrame:
    """Fix known data-quality problems. Returns a NEW DataFrame."""
    out = df.drop_duplicates(subset="customer_id").copy()
    removed = len(df) - len(out)
    out["plan"] = out["plan"].astype("string").str.strip().str.lower()
    unknown_plan = ~out["plan"].isin(VALID_PLANS)
    if unknown_plan.any():
        logger.warning("Dropping %d rows with unknown plan values", int(unknown_plan.sum()))
        out = out[~unknown_plan]
    bad_fee = out["monthly_fee"] <= 0
    out.loc[bad_fee, "monthly_fee"] = np.nan          # impossible value -> missing; imputed inside the model
    out["users"] = out["users"].clip(lower=1)          # avoid division by zero in features
    logger.info("Cleaning: removed %d duplicates, %d non-positive fees set to missing", removed, int(bad_fee.sum()))
    return out.reset_index(drop=True)
''')

2026-10-21 17:43:25,728 | INFO | wrote src/churnkit/data.py             (51 lines)


In [11]:
# Feature engineering in one place, shared by training and prediction
_ = write_file('src/churnkit/features.py', r'''
"""Feature engineering shared by training and prediction (training = serving)."""

import pandas as pd

NUMERIC_FEATURES = ["monthly_fee", "tenure_years", "fee_per_user", "support_calls"]
CATEGORICAL_FEATURES = ["plan"]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add model features. Returns a NEW DataFrame."""
    out = df.copy()
    out["tenure_years"] = out["tenure_months"] / 12
    out["fee_per_user"] = out["monthly_fee"] / out["users"]
    return out
''')

2026-10-21 17:43:25,766 | INFO | wrote src/churnkit/features.py         (14 lines)


---
## 9. The model module: train, evaluate, save, load

What changes compared with the script:
- Imputation and one-hot encoding move **inside an sklearn `Pipeline`**: they are learned on training data only (no leakage) and applied identically to new data (no `get_dummies` column mismatch).
- Evaluation reports ROC-AUC **and** the "always predict no churn" baseline accuracy, so an accuracy number cannot fool anyone.
- The quality gate is configurable and **raises** `ModelQualityError` instead of silently skipping the save.
- The model is saved with a `metrics.json` that records config, library versions and time.

In [12]:
# Model life-cycle: build, train + evaluate (with quality gate), save, load, predict
_ = write_file('src/churnkit/model.py', r'''
"""Training, evaluation and persistence of the churn model."""

import json
import logging
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import joblib
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from churnkit import __version__
from churnkit.config import TrainConfig
from churnkit.data import TARGET
from churnkit.exceptions import ArtifactNotFoundError, ModelQualityError
from churnkit.features import CATEGORICAL_FEATURES, NUMERIC_FEATURES

logger = logging.getLogger(__name__)
MODEL_FILE = "model.joblib"
METADATA_FILE = "metrics.json"


def build_pipeline(cfg: TrainConfig) -> Pipeline:
    """Preprocessing + logistic regression in one object (no leakage, training = serving)."""
    preprocess = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]),
         NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])
    model = LogisticRegression(C=cfg.C, max_iter=cfg.max_iter)
    return Pipeline([("preprocess", preprocess), ("model", model)])


def train_and_evaluate(df: pd.DataFrame, cfg: TrainConfig) -> tuple[Pipeline, dict[str, float]]:
    """Split, fit, evaluate on the held-out part, and enforce the quality gate."""
    X, y = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES], df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_size, stratify=y, random_state=cfg.random_state)
    pipeline = build_pipeline(cfg).fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    metrics = {
        "roc_auc": round(float(roc_auc_score(y_test, proba)), 4),
        "accuracy": round(float(accuracy_score(y_test, proba >= 0.5)), 4),
        "baseline_accuracy": round(float(max(y_test.mean(), 1 - y_test.mean())), 4),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
    }
    logger.info("Evaluation: %s", metrics)
    if metrics["roc_auc"] < cfg.min_auc:
        raise ModelQualityError(f"ROC-AUC {metrics['roc_auc']:.3f} is below the required {cfg.min_auc}")
    return pipeline, metrics


def save_artifacts(pipeline: Pipeline, metrics: dict[str, float], cfg: TrainConfig) -> Path:
    """Save model + metadata (config, versions, time) into cfg.output_dir."""
    out_dir = Path(cfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipeline, out_dir / MODEL_FILE)
    metadata = {
        "churnkit_version": __version__,
        "sklearn_version": sklearn.__version__,
        "trained_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "metrics": metrics,
        "config": {key: str(value) for key, value in asdict(cfg).items()},
    }
    (out_dir / METADATA_FILE).write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    logger.info("Saved model and metadata to %s", out_dir)
    return out_dir


def load_model(model_dir: Path) -> Pipeline:
    """Load a saved pipeline. Raises ArtifactNotFoundError with a clear message."""
    path = Path(model_dir) / MODEL_FILE
    if not path.exists():
        raise ArtifactNotFoundError(f"No model at {path}; run `churnkit train` first")
    return joblib.load(path)


def predict_proba(pipeline: Pipeline, df: pd.DataFrame) -> pd.Series:
    """Churn probability for each row of an already cleaned + featurised DataFrame."""
    proba = pipeline.predict_proba(df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])[:, 1]
    return pd.Series(proba, index=df.index, name="churn_probability")
''')

2026-10-21 17:43:25,813 | INFO | wrote src/churnkit/model.py            (90 lines)


---
## 10. The command-line interface: argparse, logging, exit codes

The CLI is the **only** place that:
- parses arguments (`argparse`, standard library, no extra dependency);
- configures logging (library modules only call `logging.getLogger(__name__)`; configuring logging inside a library would hijack the logging of every application that imports it);
- turns exceptions into **exit codes**.

**stdout vs stderr:** log messages go to *stderr* (for humans and log collectors). The final *result* of the command is one JSON line on *stdout* (for other programs). This is the one place where `print` is the right tool.

| Exit code | Meaning | Who reacts |
|---|---|---|
| 0 | success | - |
| 2 | bad configuration / arguments (argparse also uses 2) | the person who changed the config |
| 3 | bad input data or missing artifact | the data / CRM team |
| 4 | model quality gate failed | the data scientist |
| 1 | any *unexpected* error (Python's default for an uncaught exception, with full traceback) | the developer |

Note what we do **not** do: catch `Exception` and hide it. Unexpected errors must crash loudly with a traceback.

In [13]:
# Command-line interface: `churnkit train` and `churnkit predict`
_ = write_file('src/churnkit/cli.py', r'''
"""Command-line interface. Usage: churnkit [--log-level LEVEL] {train,predict} ..."""

import argparse
import json
import logging
import sys
from pathlib import Path

from churnkit import __version__
from churnkit.config import load_config
from churnkit.data import clean_customers, load_customers, validate_columns
from churnkit.exceptions import (ArtifactNotFoundError, ConfigError, DataValidationError,
                                 ModelQualityError)
from churnkit.features import add_features
from churnkit.model import load_model, predict_proba, save_artifacts, train_and_evaluate

logger = logging.getLogger("churnkit")

EXIT_OK, EXIT_CONFIG, EXIT_INPUT, EXIT_QUALITY = 0, 2, 3, 4


def build_parser() -> argparse.ArgumentParser:
    """Define the commands and options (also produces --help)."""
    parser = argparse.ArgumentParser(prog="churnkit", description="Train and run the churn model.")
    parser.add_argument("--version", action="version", version=f"%(prog)s {__version__}")
    parser.add_argument("--log-level", default="INFO", choices=["DEBUG", "INFO", "WARNING", "ERROR"])
    sub = parser.add_subparsers(dest="command", required=True)

    train = sub.add_parser("train", help="train a model and save it with its metrics")
    train.add_argument("--config", type=Path, default=None, help="YAML or JSON config file")

    predict = sub.add_parser("predict", help="score customers with a saved model")
    predict.add_argument("--model-dir", type=Path, required=True, help="folder created by `train`")
    predict.add_argument("--input", type=Path, required=True, help="CSV export without the target")
    predict.add_argument("--output", type=Path, required=True, help="where to write the scores CSV")
    return parser


def run_train(args: argparse.Namespace) -> int:
    """Load config and data, train, evaluate, save. Prints one JSON result line."""
    cfg = load_config(args.config)
    logger.info("Configuration: %s", cfg)
    raw = load_customers(cfg.data_path)
    validate_columns(raw, require_target=True)
    df = add_features(clean_customers(raw))
    pipeline, metrics = train_and_evaluate(df, cfg)
    out_dir = save_artifacts(pipeline, metrics, cfg)
    print(json.dumps({"status": "ok", "output_dir": str(out_dir), **metrics}))
    return EXIT_OK


def run_predict(args: argparse.Namespace) -> int:
    """Score a CSV with a saved model. Prints one JSON result line."""
    pipeline = load_model(args.model_dir)
    raw = load_customers(args.input)
    validate_columns(raw, require_target=False)
    df = add_features(clean_customers(raw))
    scores = df[["customer_id"]].assign(churn_probability=predict_proba(pipeline, df).round(4))
    args.output.parent.mkdir(parents=True, exist_ok=True)
    scores.to_csv(args.output, index=False)
    print(json.dumps({"status": "ok", "rows_scored": len(scores), "output": str(args.output)}))
    return EXIT_OK


def main(argv: list[str] | None = None) -> int:
    """Entry point. Returns an exit code instead of calling sys.exit (easier to test)."""
    args = build_parser().parse_args(argv)
    logging.basicConfig(level=args.log_level, stream=sys.stderr,
                        format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s")
    handlers = {"train": run_train, "predict": run_predict}
    try:
        return handlers[args.command](args)
    except ConfigError as exc:
        logger.error("Configuration problem: %s", exc)
        return EXIT_CONFIG
    except (DataValidationError, ArtifactNotFoundError) as exc:
        logger.error("Input problem: %s", exc)
        return EXIT_INPUT
    except ModelQualityError as exc:
        logger.error("Quality gate failed, model NOT saved: %s", exc)
        return EXIT_QUALITY


if __name__ == "__main__":
    sys.exit(main())
''')

2026-10-21 17:43:25,849 | INFO | wrote src/churnkit/cli.py              (85 lines)


In [14]:
# Makes `python -m churnkit ...` work
_ = write_file('src/churnkit/__main__.py', r'''
"""Allow `python -m churnkit`."""

import sys

from churnkit.cli import main

sys.exit(main())
''')

2026-10-21 17:43:25,877 | INFO | wrote src/churnkit/__main__.py         (7 lines)


---
## 11. Packaging metadata: `pyproject.toml`

`pyproject.toml` is the single standard file (PEP 621) that describes the project: name, version, Python version, dependencies, the **console script** (`churnkit` command, installed by `pip install .`), and settings for tools such as pytest and ruff. `02_industrial.ipynb` printed one as a string; here we write it, validate it with `tomllib`, and later actually build the package from it.

In [15]:
pyproject_path = write_file("pyproject.toml", """
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "churnkit"
version = "0.1.0"
description = "Train and run the customer-churn model (refactored from a notebook)"
requires-python = ">=3.10"
dependencies = ["pandas>=2.0", "scikit-learn>=1.3", "joblib>=1.3", "PyYAML>=6.0"]

[project.optional-dependencies]
dev = ["pytest>=8", "ruff>=0.5", "mypy>=1.10"]

[project.scripts]
churnkit = "churnkit.cli:main"          # `pip install .` creates a `churnkit` command

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]                    # tests import the package from src/ without installing it
addopts = "-ra"

[tool.ruff]
line-length = 110
""")

# Validate: a typo in TOML would break installs for the whole team
meta = tomllib.loads(pyproject_path.read_text())
print("name/version :", meta["project"]["name"], meta["project"]["version"])
print("dependencies :", meta["project"]["dependencies"])
print("CLI command  :", meta["project"]["scripts"])

print("\nProject files:")
for path in sorted(p for p in WORKDIR.rglob("*") if p.is_file() and "__pycache__" not in p.parts):
    print(f"  {str(path.relative_to(WORKDIR)):38s} {path.stat().st_size:7,d} bytes")

2026-10-21 17:43:25,918 | INFO | wrote pyproject.toml                   (27 lines)


name/version : churnkit 0.1.0
dependencies : ['pandas>=2.0', 'scikit-learn>=1.3', 'joblib>=1.3', 'PyYAML>=6.0']
CLI command  : {'churnkit': 'churnkit.cli:main'}

Project files:
  configs/train.yaml                         142 bytes
  data/customers.csv                      44,970 bytes
  data/customers_renamed.csv              44,966 bytes
  data/model.pkl                             950 bytes
  data/new_customers.csv                   9,360 bytes
  legacy/train_model.py                      938 bytes
  next_week/customers.csv                 44,966 bytes
  next_week/model.pkl                        950 bytes
  pyproject.toml                             755 bytes
  src/churnkit/__init__.py                    79 bytes
  src/churnkit/__main__.py                    95 bytes
  src/churnkit/cli.py                      3,782 bytes
  src/churnkit/config.py                   3,125 bytes
  src/churnkit/data.py                     2,321 bytes
  src/churnkit/exceptions.py                 610 byte

---
## 12. Running the CLI like a scheduler would

Everything below runs in a **separate process** with `subprocess`, exactly as cron, Airflow, a CI job or a colleague's terminal would run it. In real life you would `pip install -e .` once and then type `churnkit train ...`; here we set `PYTHONPATH=src` so nothing is installed into the course environment.

In [16]:
# 1. Self-documentation: argparse generates --help for free
_ = run([PYTHON, "-m", "churnkit", "--help"])

$ python -m churnkit --help
-> exit code 0
[stdout]
usage: churnkit [-h] [--version] [--log-level {DEBUG,INFO,WARNING,ERROR}]
                {train,predict} ...

Train and run the churn model.

positional arguments:
  {train,predict}
    train               train a model and save it with its metrics
    predict             score customers with a saved model

options:
  -h, --help            show this help message and exit
  --version             show program's version number and exit
  --log-level {DEBUG,INFO,WARNING,ERROR}


In [17]:
# 2. Train with the reviewed YAML config
train_result = run([PYTHON, "-m", "churnkit", "train", "--config", "configs/train.yaml"])
metadata = json.loads((WORKDIR / "artifacts/churn/v1/metrics.json").read_text())
print("\nmetrics.json ->", json.dumps(metadata["metrics"]), "| sklearn", metadata["sklearn_version"])

$ python -m churnkit train --config configs/train.yaml
-> exit code 0
[stdout]
{"status": "ok", "output_dir": "artifacts/churn/v1", "roc_auc": 0.7836, "accuracy": 0.78, "baseline_accuracy": 0.7433, "n_train": 1200, "n_test": 300}
[stderr]
2026-10-21 17:43:36,051 | INFO    | churnkit | Configuration: TrainConfig(data_path=PosixPath('data/customers.csv'), output_dir=PosixPath('artifacts/churn/v1'), test_size=0.2, random_state=42, C=0.5, max_iter=1000, min_auc=0.65)
2026-10-21 17:43:36,075 | INFO    | churnkit.data | Loaded 1525 rows x 7 columns from customers.csv
2026-10-21 17:43:36,104 | INFO    | churnkit.data | Cleaning: removed 25 duplicates, 12 non-positive fees set to missing
2026-10-21 17:43:36,306 | INFO    | churnkit.model | Evaluation: {'roc_auc': 0.7836, 'accuracy': 0.78, 'baseline_accuracy': 0.7433, 'n_train': 1200, 'n_test': 300}
2026-10-21 17:43:36,312 | INFO    | churnkit.model | Saved model and metadata to artifacts/churn/v1

metrics.json -> {"roc_auc": 0.7836, "accuracy"

Compare with the legacy script: the logs now say *what* was cleaned (duplicates, bad fees), the metrics include ROC-AUC and the "always no churn" baseline accuracy, and the result line on stdout is JSON that another program can parse. Note that the model's accuracy is only a few points above that baseline: that is exactly why the quality gate uses ROC-AUC, not accuracy.

In [18]:
# 3. A one-off experiment WITHOUT editing any file: override settings via environment variables
_ = run([PYTHON, "-m", "churnkit", "--log-level", "WARNING", "train", "--config", "configs/train.yaml"],
        extra_env={"CHURNKIT_C": "0.05", "CHURNKIT_OUTPUT_DIR": "artifacts/churn/exp_small_C"})

$ CHURNKIT_C=0.05 CHURNKIT_OUTPUT_DIR=artifacts/churn/exp_small_C python -m churnkit --log-level WARNING train --config configs/train.yaml
-> exit code 0
[stdout]
{"status": "ok", "output_dir": "artifacts/churn/exp_small_C", "roc_auc": 0.7822, "accuracy": 0.7733, "baseline_accuracy": 0.7433, "n_train": 1200, "n_test": 300}


In [19]:
# 4. Batch scoring with the saved model
_ = run([PYTHON, "-m", "churnkit", "predict", "--model-dir", "artifacts/churn/v1",
         "--input", "data/new_customers.csv", "--output", "outputs/scores.csv"])
pd.read_csv(WORKDIR / "outputs/scores.csv").sort_values("churn_probability", ascending=False).head()

$ python -m churnkit predict --model-dir artifacts/churn/v1 --input data/new_customers.csv --output outputs/scores.csv
-> exit code 0
[stdout]
{"status": "ok", "rows_scored": 300, "output": "outputs/scores.csv"}
[stderr]
2026-10-21 17:43:45,047 | INFO    | churnkit.data | Loaded 325 rows x 6 columns from new_customers.csv
2026-10-21 17:43:45,053 | INFO    | churnkit.data | Cleaning: removed 25 duplicates, 9 non-positive fees set to missing


,customer_id,churn_probability
35,C1100035,0.8414
1,C1100001,0.7652
183,C1100183,0.7613
232,C1100232,0.6939
52,C1100052,0.6894


### The failure that hurt the team, replayed
Same renamed-column export as in Section 3. A good production job **fails fast, says why, and returns a non-zero exit code** so the scheduler marks the run red and alerts someone.

In [20]:
failures = {
    "renamed column": run([PYTHON, "-m", "churnkit", "train", "--config", "configs/train.yaml"],
                          extra_env={"CHURNKIT_DATA_PATH": "data/customers_renamed.csv"}),
    "config typo": run([PYTHON, "-m", "churnkit", "train", "--config", "configs/train.yaml"],
                       extra_env={"CHURNKIT_TEST_SIZE": "20"}),
    "quality gate": run([PYTHON, "-m", "churnkit", "train", "--config", "configs/train.yaml"],
                        extra_env={"CHURNKIT_MIN_AUC": "0.95", "CHURNKIT_OUTPUT_DIR": "artifacts/should_not_exist"},
                        quiet=True),
    "unknown command": run([PYTHON, "-m", "churnkit", "deploy"], quiet=True),
}
summary = pd.DataFrame({"exit_code": {k: r.returncode for k, r in failures.items()},
                        "last_message": {k: (r.stderr.strip().splitlines() or [""])[-1].split("| ")[-1][:85]
                                         for k, r in failures.items()}})
print("\nModel saved despite failed quality gate?", (WORKDIR / "artifacts/should_not_exist").exists())
summary

$ CHURNKIT_DATA_PATH=data/customers_renamed.csv python -m churnkit train --config configs/train.yaml
-> exit code 3
[stderr]
2026-10-21 17:43:50,952 | INFO    | churnkit | Configuration: TrainConfig(data_path=PosixPath('data/customers_renamed.csv'), output_dir=PosixPath('artifacts/churn/v1'), test_size=0.2, random_state=42, C=0.5, max_iter=1000, min_auc=0.65)
2026-10-21 17:43:50,959 | INFO    | churnkit.data | Loaded 1525 rows x 7 columns from customers_renamed.csv
2026-10-21 17:43:50,960 | ERROR   | churnkit | Input problem: Missing required column(s) ['tenure_months']; got ['churned', 'customer_id', 'monthly_fee', 'plan', 'support_calls', 'tenure_mo', 'users']


$ CHURNKIT_TEST_SIZE=20 python -m churnkit train --config configs/train.yaml
-> exit code 2
[stderr]
2026-10-21 17:43:55,663 | ERROR   | churnkit | Configuration problem: test_size must be between 0 and 1, got 20.0


$ CHURNKIT_MIN_AUC=0.95 CHURNKIT_OUTPUT_DIR=artifacts/should_not_exist python -m churnkit train --config configs/train.yaml
-> exit code 4


$ python -m churnkit deploy
-> exit code 2

Model saved despite failed quality gate? False


,exit_code,last_message
renamed column,3,Input problem: Missing required column(s) ['te...
config typo,2,Configuration problem: test_size must be betwe...
quality gate,4,"Quality gate failed, model NOT saved: ROC-AUC ..."
unknown command,2,churnkit: error: argument command: invalid cho...


Every failure now has its own exit code and a one-line explanation, and the failed quality gate did **not** leave a model behind. Compare with the legacy script, which printed `saved` and exited with 0 for the renamed column.

---
## 13. Unit tests with pytest (really executed)

Tests live in `tests/` and are discovered automatically by **pytest** (functions named `test_*`). Features we use:
- **fixtures** (`conftest.py`): reusable test inputs, e.g. a tiny messy DataFrame, or a CSV in pytest's temporary folder `tmp_path`;
- **`pytest.raises`**: check that bad input raises the *right* error with the *right* message;
- **`parametrize`**: run the same test for several inputs;
- **`monkeypatch`**: set environment variables only for one test;
- an **end-to-end test** that calls `main([...])` directly (that is why `main` returns an exit code instead of calling `sys.exit`).

Fast tests (seconds) run on every commit in CI (GitHub Actions, GitLab CI); see `20_Git_and_DevOps/`.

In [21]:
# Shared fixtures for all test files
_ = write_file('tests/conftest.py', r'''
"""Shared pytest fixtures."""

from pathlib import Path

import numpy as np
import pandas as pd
import pytest


@pytest.fixture
def raw_customers() -> pd.DataFrame:
    """A tiny, hand-made messy export: a duplicate, odd spelling, a negative fee, zero users."""
    return pd.DataFrame({
        "customer_id": ["A", "A", "B", "C", "D"],
        "plan": [" PRO ", " PRO ", "basic", "Enterprise", "trial"],
        "monthly_fee": [60.0, 60.0, -1.0, 150.0, 10.0],
        "tenure_months": [12, 12, 3, 40, 1],
        "users": [2, 2, 0, 10, 1],
        "support_calls": [1, 1, 4, 0, 0],
        "churned": [0, 0, 1, 0, 1],
    })


@pytest.fixture
def customers_csv(tmp_path: Path) -> Path:
    """A small but learnable dataset written to pytest's temporary folder."""
    rng = np.random.default_rng(0)
    n = 400
    calls = rng.poisson(1.5, n)
    tenure = rng.integers(1, 60, n)
    churned = rng.binomial(1, 1 / (1 + np.exp(-(-0.5 + 0.8 * calls - 0.06 * tenure))))
    df = pd.DataFrame({"customer_id": [f"T{i}" for i in range(n)],
                       "plan": rng.choice(["basic", "pro", "enterprise"], n),
                       "monthly_fee": rng.uniform(15, 150, n).round(2), "tenure_months": tenure,
                       "users": rng.integers(1, 10, n), "support_calls": calls, "churned": churned})
    path = tmp_path / "customers.csv"
    df.to_csv(path, index=False)
    return path
''')

2026-10-21 17:44:05,003 | INFO | wrote tests/conftest.py                (38 lines)


In [22]:
# Configuration: precedence, validation, clear errors
_ = write_file('tests/test_config.py', r'''
"""Tests for churnkit.config."""

import json

import pytest

from churnkit.config import TrainConfig, load_config
from churnkit.exceptions import ConfigError


def test_defaults_are_valid():
    assert load_config(env={}) == TrainConfig()


def test_yaml_file_overrides_defaults(tmp_path):
    path = tmp_path / "cfg.yaml"
    path.write_text("C: 0.5\nmax_iter: 50\n")
    cfg = load_config(path, env={})
    assert cfg.C == 0.5 and cfg.max_iter == 50 and cfg.test_size == 0.2


def test_environment_overrides_file(tmp_path):
    path = tmp_path / "cfg.json"
    path.write_text(json.dumps({"C": 0.5}))
    assert load_config(path, env={"CHURNKIT_C": "3"}).C == 3.0      # string from env -> float


@pytest.mark.parametrize("setting, value", [("test_size", 1.5), ("C", -1), ("min_auc", 0.2), ("max_iter", 0)])
def test_invalid_values_are_rejected(tmp_path, setting, value):
    path = tmp_path / "cfg.json"
    path.write_text(json.dumps({setting: value}))
    with pytest.raises(ConfigError, match=setting):
        load_config(path, env={})


def test_unknown_setting_is_rejected(tmp_path):
    path = tmp_path / "cfg.yaml"
    path.write_text("learning_rate: 0.1\n")              # typo / setting that does not exist
    with pytest.raises(ConfigError, match="Unknown setting"):
        load_config(path, env={})


def test_bad_type_in_env_has_clear_message():
    with pytest.raises(ConfigError, match="max_iter"):
        load_config(env={"CHURNKIT_MAX_ITER": "many"})


def test_missing_config_file(tmp_path):
    with pytest.raises(ConfigError, match="not found"):
        load_config(tmp_path / "nope.yaml", env={})
''')

2026-10-21 17:44:05,034 | INFO | wrote tests/test_config.py             (50 lines)


In [23]:
# Data functions: cleaning rules and the data contract
_ = write_file('tests/test_data.py', r'''
"""Tests for churnkit.data and churnkit.features."""

import pandas as pd
import pytest

from churnkit.data import clean_customers, load_customers, validate_columns
from churnkit.exceptions import DataValidationError
from churnkit.features import add_features


def test_clean_removes_duplicates_and_unknown_plans(raw_customers):
    clean = clean_customers(raw_customers)
    assert clean["customer_id"].tolist() == ["A", "B", "C"]          # duplicate A and plan 'trial' removed
    assert set(clean["plan"]) == {"pro", "basic", "enterprise"}      # normalised spelling


def test_clean_does_not_modify_its_input(raw_customers):
    before = raw_customers.copy()
    clean_customers(raw_customers)
    pd.testing.assert_frame_equal(raw_customers, before)


def test_non_positive_fee_becomes_missing_and_users_are_clipped(raw_customers):
    clean = clean_customers(raw_customers).set_index("customer_id")
    assert pd.isna(clean.loc["B", "monthly_fee"])
    assert clean.loc["B", "users"] == 1


@pytest.mark.parametrize("dropped_column", ["tenure_months", "plan", "churned"])
def test_missing_column_fails_loudly(raw_customers, dropped_column):
    with pytest.raises(DataValidationError, match=dropped_column):
        validate_columns(raw_customers.drop(columns=dropped_column))


def test_prediction_data_does_not_need_target(raw_customers):
    validate_columns(raw_customers.drop(columns="churned"), require_target=False)   # must not raise


def test_missing_file_raises_data_error(tmp_path):
    with pytest.raises(DataValidationError, match="not found"):
        load_customers(tmp_path / "missing.csv")


def test_features_are_computed():
    feats = add_features(pd.DataFrame({"tenure_months": [24], "monthly_fee": [60.0], "users": [3]}))
    assert feats.loc[0, "tenure_years"] == 2 and feats.loc[0, "fee_per_user"] == 20
''')

2026-10-21 17:44:05,060 | INFO | wrote tests/test_data.py               (46 lines)


In [24]:
# End-to-end: the CLI exactly as users and schedulers call it
_ = write_file('tests/test_cli.py', r'''
"""End-to-end tests through churnkit.cli.main."""

import json

import pandas as pd

from churnkit.cli import EXIT_CONFIG, EXIT_INPUT, EXIT_OK, EXIT_QUALITY, main


def _config(tmp_path, data_path, **extra):
    path = tmp_path / "cfg.json"
    path.write_text(json.dumps({"data_path": str(data_path), "output_dir": str(tmp_path / "out"), **extra}))
    return path


def test_train_then_predict(customers_csv, tmp_path, monkeypatch, capsys):
    monkeypatch.delenv("CHURNKIT_C", raising=False)
    assert main(["train", "--config", str(_config(tmp_path, customers_csv))]) == EXIT_OK
    result = json.loads(capsys.readouterr().out.strip().splitlines()[-1])     # stdout = machine-readable
    assert result["status"] == "ok" and result["roc_auc"] > 0.65
    scores_path = tmp_path / "scores.csv"
    code = main(["predict", "--model-dir", str(tmp_path / "out"), "--input", str(customers_csv),
                 "--output", str(scores_path)])
    assert code == EXIT_OK
    scores = pd.read_csv(scores_path)
    assert scores["churn_probability"].between(0, 1).all() and len(scores) == 400


def test_missing_input_file_returns_input_error(tmp_path):
    assert main(["train", "--config", str(_config(tmp_path, tmp_path / "nope.csv"))]) == EXIT_INPUT


def test_env_override_reaches_the_cli(customers_csv, tmp_path, monkeypatch):
    monkeypatch.setenv("CHURNKIT_TEST_SIZE", "7")                            # invalid on purpose
    assert main(["train", "--config", str(_config(tmp_path, customers_csv))]) == EXIT_CONFIG


def test_quality_gate_blocks_saving(customers_csv, tmp_path):
    cfg = _config(tmp_path, customers_csv, min_auc=0.99)
    assert main(["train", "--config", str(cfg)]) == EXIT_QUALITY
    assert not (tmp_path / "out" / "model.joblib").exists()
''')

2026-10-21 17:44:05,088 | INFO | wrote tests/test_cli.py                (41 lines)


In [25]:
# Run the whole test suite in a separate process, like CI does
pytest_result = None
if importlib.util.find_spec("pytest") is None:
    print("pytest is not installed in this environment. Install it with:  pip install pytest==9.1.1")
else:
    pytest_result = run([PYTHON, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"])
    print("\nTest suite", "PASSED" if pytest_result.returncode == 0 else "FAILED")

$ python -m pytest -q --color=no -p no:cacheprovider
-> exit code 0
[stdout]
.......................                                                  [100%]
23 passed in 2.62s

Test suite PASSED


The final line of pytest's output shows how many tests passed and how long they took. A suite that runs in seconds gets run on every change; a suite that takes an hour gets skipped. Keep unit tests small and fast, and keep a few end-to-end tests (like `test_train_then_predict`) for the full path.

> **Try it:** break something on purpose, e.g. change `.clip(lower=1)` to `.clip(lower=0)` in `data.py`, re-run the `write_file` cell for `data.py` and then this cell. Exactly one test should fail and its message tells you which rule you broke. That is the safety net that makes refactoring possible.

---
## 14. Building a wheel (proof that the package is installable)

A **wheel** (`.whl`) is the file `pip` installs. Building one checks that `pyproject.toml`, the `src/` layout and the entry point are consistent. We build it in the workspace without installing it (`--no-deps`, `--no-build-isolation` use the setuptools already available, so nothing is downloaded).

In [26]:
import zipfile                                               # a wheel is just a zip file

wheel_result = run([PYTHON, "-m", "pip", "wheel", ".", "--no-deps", "--no-build-isolation", "-w", "dist", "-q"],
                   quiet=True)
wheels = sorted((WORKDIR / "dist").glob("*.whl"))
if wheels:
    with zipfile.ZipFile(wheels[0]) as wheel:
        names = wheel.namelist()
        entry_points = next(n for n in names if n.endswith("entry_points.txt"))
        print("Built:", wheels[0].name)
        print("Package files:", [n for n in names if n.startswith("churnkit/")])
        print("entry_points.txt:\n" + wheel.read(entry_points).decode())
else:
    print("Wheel build failed (see stderr above); this does not affect the rest of the notebook.")
    print(wheel_result.stderr[-1500:])

$ python -m pip wheel . --no-deps --no-build-isolation -w dist -q
-> exit code 0
Built: churnkit-0.1.0-py3-none-any.whl
Package files: ['churnkit/__init__.py', 'churnkit/__main__.py', 'churnkit/cli.py', 'churnkit/config.py', 'churnkit/data.py', 'churnkit/exceptions.py', 'churnkit/features.py', 'churnkit/model.py']
entry_points.txt:
[console_scripts]
churnkit = churnkit.cli:main



Notice what is **not** in the wheel: tests, configs, data, artifacts. A package ships code; data and models are delivered separately (data lake, model registry).

---
## 15. Automated before/after review

Linters such as **ruff** (rules `E722` bare except, `T201` print) and type checkers such as **mypy** automate much of a code review. To see the idea without installing anything, here is a tiny reviewer built on Python's own `ast` module (it reads code as a tree, the same way linters do).

In [27]:
def review_source(paths: list[Path]) -> dict:
    """Count a few code-quality signals across Python files using the ast module."""
    stats = {"files": 0, "lines": 0, "functions": 0, "functions_fully_typed": 0, "functions_with_docstring": 0,
             "print_calls": 0, "bare_except": 0, "top_level_statements": 0, "hard_coded_file_names": 0}
    for path in paths:
        source = path.read_text()
        tree = ast.parse(source)
        stats["files"] += 1
        stats["lines"] += len(source.splitlines())
        for node in tree.body:     # statements that run on import (not defs, classes, imports, docstrings, constants)
            is_constant = isinstance(node, ast.Assign) and isinstance(node.value, (ast.Constant, ast.Tuple, ast.List, ast.Call)) \
                and all(isinstance(t, (ast.Name, ast.Tuple)) and getattr(t, "id", "X").isupper() for t in node.targets)
            is_main_guard = isinstance(node, ast.If) and "__name__" in ast.unparse(node.test)
            if not isinstance(node, (ast.FunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom)) \
                    and not (isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant)) \
                    and not is_constant and not is_main_guard and path.name != "__main__.py" \
                    and not (isinstance(node, ast.Assign) and ast.unparse(node).startswith(("logger", "__version__"))):
                stats["top_level_statements"] += 1
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                stats["functions"] += 1
                args = [a for a in node.args.args if a.arg not in ("self", "cls")]
                stats["functions_fully_typed"] += int(node.returns is not None and all(a.annotation for a in args))
                stats["functions_with_docstring"] += int(ast.get_docstring(node) is not None)
            elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "print":
                stats["print_calls"] += 1
            elif isinstance(node, ast.ExceptHandler) and node.type is None:
                stats["bare_except"] += 1
            elif isinstance(node, ast.Constant) and isinstance(node.value, str) \
                    and node.value.endswith((".csv", ".pkl", ".joblib")):
                stats["hard_coded_file_names"] += 1
    return stats


review = pd.DataFrame({
    "before: legacy script": review_source([WORKDIR / "legacy/train_model.py"]),
    "after: src/churnkit": review_source(sorted((WORKDIR / "src/churnkit").glob("*.py"))),
    "after: tests": review_source(sorted((WORKDIR / "tests").glob("*.py"))),
})
review

,before: legacy script,after: src/churnkit,after: tests
files,1,8,4
lines,27,352,175
functions,0,17,21
functions_fully_typed,0,17,2
functions_with_docstring,0,16,2
print_calls,3,2,0
bare_except,1,0,0
top_level_statements,14,0,0
hard_coded_file_names,2,2,5


**How to read it:**
- The refactor has **more lines**: that is normal and fine. We traded a short script for validation, error handling, docs and configuration. Lines are not the cost; *unreadable* lines are.
- The legacy script is almost entirely **top-level statements** (code that runs on import, impossible to test piece by piece); the package has (almost) none.
- `print` survives only where it belongs: the two JSON result lines of the CLI.
- No bare `except`; all 17 package functions are fully typed and 16 have a docstring (the exception is `__post_init__`, whose error messages explain the rules).
- The remaining "hard-coded file names" in the package are named defaults/constants (`model.joblib`, the default `data_path`), which is fine: they are defined **once** and can be overridden.
- Tests are not typed or documented as strictly: common practice; test names are the documentation.

---
## 16. Code-review checklist

Use this in pull requests (copy it into your PR template). Items marked *auto* can be enforced by tools in CI.

| Area | Question | Tool |
|---|---|---|
| Structure | Does each function do one thing, and is it under ~40 lines? | review |
| Structure | Does code run only inside functions (no work at import time), with a `main()` entry point? | auto: our `ast` check |
| Naming & types | Are names descriptive, and are public functions type-hinted? | auto: mypy / pyright |
| Docs | Do modules and public functions have docstrings that say *why*, not just *what*? | auto: ruff `D` rules |
| Config | Are there any magic numbers / paths / secrets in code instead of config? | review + secret scanners |
| Errors | Are errors specific (custom exceptions) and never silently swallowed (`except: pass`)? | auto: ruff `E722`, `BLE` |
| Logging | `logging` for diagnostics (no logging config in library code), stdout only for results? | auto: ruff `T201` |
| Data | Are inputs validated at the boundary (files, APIs) with clear messages? | review + tests |
| ML specifics | Preprocessing inside the pipeline (no leakage)? Seeds fixed? Metadata saved with the model? | review |
| Tests | Is there a test for the change, including the failure path? Do they run in seconds? | auto: pytest + coverage in CI |
| CLI | Does `--help` explain usage? Non-zero exit code on every failure? | auto: CLI tests |
| Packaging | Are dependencies declared in `pyproject.toml` (and pinned in a lock file for deployments)? | auto: build in CI |
| Behaviour | Did the refactor change results? (compare metrics before/after on the same data) | auto: regression test |

The last row matters most during a refactor: our package uses the same model family as the legacy script (logistic regression, same features), but fixes leakage and the train/serve mismatch, so small metric differences are expected and should be **explained in the PR**, not hidden.

In [28]:
# The "behaviour" row in practice: compare the legacy result with the refactored one on the same data
import re                                                     # regular expressions, to read pytest's summary line
legacy_accuracy = float(before_ok.stdout.split("acc")[-1].split()[0])
new_metrics = json.loads(train_result.stdout.strip().splitlines()[-1])
comparison = pd.DataFrame({
    "legacy script": {"accuracy": round(legacy_accuracy, 4), "roc_auc": "not computed", "fails on bad input": "no (silent)",
                      "tests": 0},
    "churnkit 0.1.0": {"accuracy": new_metrics["accuracy"], "roc_auc": new_metrics["roc_auc"],
                       "fails on bad input": "yes (exit code 3)",
                       "tests": int(re.search(r"(\d+) passed", pytest_result.stdout).group(1)) if pytest_result else 0},
})
comparison

,legacy script,churnkit 0.1.0
accuracy,0.7248,0.78
roc_auc,not computed,0.7836
fails on bad input,no (silent),yes (exit code 3)
tests,0,23


The refactored version scores higher accuracy (0.78 vs 0.72). Part of that is real (features are now imputed and **scaled** inside the pipeline, which logistic regression needs; the legacy model was even below the 0.74 "always no churn" baseline), and part may be the different random split. A proper PR would re-run both versions on the *same* split and report the difference; the point here is that the new version also reports ROC-AUC, refuses broken input, and is covered by 23 tests.

---
## 17. Summary, exercises and resources

| Practice | What we did | Where it is used in industry |
|---|---|---|
| Small typed functions | `load_customers`, `validate_columns`, `clean_customers`, `add_features`... | Every maintainable code base |
| `src/` layout + `pyproject.toml` | Real files, validated with `tomllib`, built into a wheel | Python Packaging Authority standard |
| Config dataclass | defaults < YAML/JSON < env vars, validated in `__post_init__` | Twelve-Factor apps, ML training jobs, Hydra/pydantic-settings |
| Logging vs stdout | logs on stderr, one JSON result line on stdout | CLIs used by schedulers and pipelines |
| Custom exceptions + exit codes | `ConfigError` -> 2, `DataValidationError` -> 3, `ModelQualityError` -> 4 | Airflow/cron/Kubernetes jobs detect and route failures |
| argparse CLI | `train` / `predict` subcommands, `--help`, `--version` | Internal tools, batch jobs |
| pytest | fixtures, parametrize, raises, monkeypatch, end-to-end test | CI on every pull request |
| Automated review | `ast`-based checks (what ruff/mypy do at scale) | Pre-commit hooks, CI linters |

### Practice exercises
1. Add a `--dry-run` flag to `train` that validates config and data but does not train. Write a test for it.
2. Add an `evaluate` subcommand that loads a saved model and scores a labelled CSV. Which exit code should a missing model return?
3. Replace the hand-written `_convert` in `config.py` with **pydantic-settings** (see `02_industrial.ipynb`, section 1). What do you gain, what dependency do you add?
4. Make the CLI write logs as JSON (one object per line) when `--log-format json` is passed. Why do log platforms prefer this?
5. Install `ruff` in a separate virtual environment and run `ruff check legacy/ src/` in the workspace. Compare its findings with Section 15.

### Resources (checked)
- Python Packaging User Guide, *Writing your pyproject.toml*: https://packaging.python.org/en/latest/guides/writing-pyproject-toml/
- *src layout vs flat layout* (PyPA): https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/
- pytest documentation (fixtures, parametrize, monkeypatch, good practices): https://docs.pytest.org/en/stable/explanation/goodpractices.html
- argparse tutorial (official): https://docs.python.org/3/howto/argparse.html
- Logging HOWTO, *Configuring logging for a library*: https://docs.python.org/3/howto/logging.html#configuring-logging-for-a-library
- The Twelve-Factor App, *III. Config*: https://12factor.net/config
- Google Python Style Guide: https://google.github.io/styleguide/pyguide.html
- Ruff linter rules (E722, T201, ...): https://docs.astral.sh/ruff/rules/
- Cookiecutter Data Science (project structure for data/ML projects): https://github.com/drivendataorg/cookiecutter-data-science
- *Refactoring* (Martin Fowler), catalog of refactorings: https://refactoring.com/catalog/

### Related notebooks in this repo
- `02_industrial.ipynb`: each production pattern (decorators, context managers, logging, pytest, packaging) explained on its own
- `03_advanced.ipynb`: protocols, descriptors and other advanced features used in frameworks
- `../03_Machine_Learning/01_supervised/14_industry_churn_prediction.ipynb`: the full churn-modelling life-cycle (the ML side of this script)
- `../15_MLOps/`: experiment tracking, model registry, testing in ML
- `../20_Git_and_DevOps/`: running these tests automatically in CI